In [1]:
import sympy as sp
from sympy import *
init_printing(use_unicode=True)
import random
import numpy as np
import math
import pandas as pd

# RKPW novo

In [2]:
def rkpw(Lambda, c):
    n = len(c)
    a = [1.0, Lambda[0]]
    b = [c[0]]
    for j in range(2, n + 1):
        lam_j = Lambda[j - 1]
        a.append(lam_j)
        b.append(0.0)
        delta1 = c[j - 1]
        delta2 = 0.0
        for i in range(1, j):
            if (b[i - 1] != 0.0 or delta1 !=0.0):
                r = sqrt(b[i-1]*b[i-1] + delta1*delta1) # usar inv_r  = 1.0/r e dividir menos vezes?
                gamma = b[i - 1]/r
                sigma = -delta1/r
                
                gammasqr = gamma*gamma
                sigmasqr = sigma*sigma
                gs = gamma*sigma
                
                b[i - 1] = r #r = b[i-1]*gamma - sigma*delta1 #b_{i-1} da J_j: obtido pela conj por G_{i+1,j+1}
                delta1 = (gammasqr - sigmasqr)*delta2 + gs*(a[i] - a[j]) #delta_1: futuro b_{j-1}
                a_j = a[j] #var provisória
                a[j] = a[j]*gammasqr + 2.0*gs*delta2 + a[i]*sigmasqr #lambda^: antigo lamda_j, futuro a_j
                a[i] = a[i]*gammasqr - 2.0*gs*delta2 + a_j*sigmasqr #a_i da J_j: obtido pela conj por G_{i+1,j+1}
                delta2 = b[i]*sigma #delta_2: antigo zero e futuro nada
                b[i] *= gamma #b[i]*gamma #b_i provisório: obtido pela conj por G_{i+1,j+1}
        b[j - 1] = delta1   
    T = zeros(n) #monta a resposta: J = J_n tridiagonal simétrica
    for i in range(n):
        T[i, i] = a[i + 1]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i + 1]
    return T

# INVBI novo (Cramer)

In [3]:
def invbi_cramer(Lambda, beta):
    n = len(Lambda)
    a = [Lambda[0]]
    b = []
    w = [1.0] #última linha da L^-1
    u = 1.0 #norma de w = entrada do canto da U...
    for i in range(1, n): #vamos de T_i(a[0],...,a[i-1],b[0],...,b[i-2]) para T_{i+1}(a'[0],...,a'[i],b'[0],...,b'[i-1])
        alpha = beta[i - 1]*u 
        alpha2 = alpha*alpha
        lam_i = Lambda[i]
        B = [1]*i #B é uma lista onde a k-ésima entrada é o produto b[k]^2...b[i-2]^2 (b's antigos!)
        for k in range(i - 2, -1, -1):
            B[k] = B[k + 1]*b[k]*b[k]
        b2_prod = B[0] # = prod(b[k]**2, k=0..i-2)
        det = prod(Lambda[k] - lam_i for k in range(i)) #determinante de (T_i - lambda[i]*Id)
        det2 = det*det # Scalar products reused across the loop
        
        Da, Db = 0.0, 1.0 # Recurrence seeds
        Dc = a[0] - lam_i #Da,Db e Dc são os menores principais de (T_i - lambda[i]Id): os Deltas que obedecem uma recursão de segunda ordem
        Sa = det2
        Sb = det2 + alpha2*b2_prod #Sa,Sb e Sc são as sominhas que obedecem uma recursão de primeira ordem
        
        a.append(lam_i) #nova entrada diagonal: futura a'[i]
        b.append(alpha) #nova entrada diagonal: futura a'[i]
        a[0] -= alpha2*b2_prod*Dc/Sb #atualizando a primeira entrada diagonal: a'[0]
        for j in range(1, i): #atualizando: a'[1],b'[0],...,a'[i-1],b'[i-2] (só roda quando i>1)
            Dc2 = Dc*Dc
            Sc = Sb + alpha2*B[j]*Dc2 #rec de 1a ordem da sominha
            
            Da, Db = Db, Dc
            aj_shift = a[j] - a[-1]             # a[j] - lam_i (a[-1] not yet changed)
            Dc = Db*aj_shift - Da*b[j - 1]*b[j - 1] #rec de 2a ordem dos menores
            
            inv_Sb = 1.0/Sb
            a[j] += alpha2*Db*B[j]*((aj_shift*Db - Dc)*inv_Sb - Dc/Sc) #aj_shift*Db - Dc = Da*b[j-1]**2
            b[j - 1] *= sqrt(Sc*Sa)*inv_Sb
            
            Sa, Sb = Sb, Sc
            
        a[i] += alpha2*det*Db/Sb #a'[i]
        b[i - 1] = alpha*sqrt(Sa*det2)/Sb #b'[i-1]
        
        for j in range(len(w)): #preparando para a próxima iteração: atualizando w,u,alpha... # Update weight vector and its norm
            w[j] = beta[i - 1]*w[j]/(lam_i - Lambda[j])
        w.append(1)
        u = sqrt(sum(w[k]*w[k] for k in range(len(w)-1)) + 1.0)  #u = sqrt(sum(w[k]**2 for k in range(len(w))))
    #print(alpha)    
    T = zeros(n) #monta a resposta: T = T_n tridiagonal simétrica
    for i in range(n):
        T[i, i] = a[i]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i]
    return T

# INVBI (Gaussian)

In [4]:
def invbi_gauss(Lambda, beta): 
    n = len(Lambda)
    w = [1.0]
    u = 1.0
    a = [Lambda[0]]
    b = []
    for i in range(2, n + 1):
        # 1. Prepare Parameters (alpha uses u from previous iteration)
        alpha = beta[i - 2]*u
        Sd = a + [Lambda[i - 1]]
        Soff = b + [alpha]
        # 2. Gaussian Elimination 
        h = []
        f = []
        # Forward elimination
        x = a[0] - Lambda[i - 1]
        h.append(x)
        for j in range(1, i - 1):
            y = b[j - 1] / h[j - 1]
            f.append(y)
            x = (a[j] - Lambda[i - 1]) - b[j - 1]*y
            h.append(x)
        # Backward substitution
        v = [0.0] * (i - 1)
        v[-1] = 1.0/h[-1]
        for j in range(i - 3, -1, -1):
            v[j] = -f[j]*v[j+1]
        v = [val * alpha for val in v]
        # 3. Tzao Update
        n1 = 1.0
        n2 = n1 + v[0]**2
        x = Soff[0]/n2
        new_a = []
        new_b = []
        if i == 2:
            p = v[0]*x
            new_a = [Sd[0] - p, Sd[1] + p]
            new_b = [x]
        else:
            new_a.append(Sd[0] + x*v[0]*v[1])
            n3 = n2 + v[1]**2
            new_b.append(x*sqrt(n3))
            y = Soff[1]/n3
            for j in range(1, i - 2):
                new_a.append(Sd[j] + v[j]*(v[j + 1]*y - v[j - 1]*x))
                n1, n2 = n2, n3
                n3 = n3 + v[j + 1]**2
                new_b.append(sqrt(n1*n3)*y)
                x = y
                y = Soff[j + 1]/n3
            new_a.append(Sd[i - 2] - v[i - 2]*(v[i - 3]*x + y))
            new_b.append(sqrt(n2)*y)
            new_a.append(Sd[i - 1] + v[i - 2]*y)
        a, b = new_a, new_b
        for j in range(i - 1):
            w[j] = beta[i - 2]*w[j]/(Lambda[i - 1] - Lambda[j])
        w.append(1.0)
        sum_sq = sum(val**2 for val in w)
        u = sqrt(sum_sq)
    # 5. monta a resposta: T = T_n tridiagonal simétrica
    T = zeros(n) 
    for i in range(n):
        T[i, i] = a[i]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i]
    return T

In [5]:
def deltas(Lambda):
    return [Lambda[i] - Lambda[i - 1] for i in range(1, len(Lambda))]

In [6]:
def sum_sqrs(v, k):
    s = 0
    for i in range(k):
        s += v[i] ** 2
    return s

In [7]:
def mosertobeta(Lambda, c):
    """Convert Moser vector c to bidiagonal coordinates beta.
    Works on a COPY of c — does not mutate the original."""
    c = c[:]                          # defensive copy: normalisation is in-place
    n_val = sqrt(sum_sqrs(c, len(c)))
    for i in range(len(c)):
        c[i] = c[i] / n_val
    d = deltas(Lambda)
    beta = []
    a = [d[0]]
    b = 1
    w = prod(a) / b
    beta.append(w * c[1] / c[0])
    for i in range(2, len(Lambda)):
        b = prod(a)
        for j in range(len(a)):
            a[j] = a[j] + d[i - 1]
        a.append(d[i - 1])
        w = prod(a) / b
        beta.append(w * c[i] / c[i - 1])
    return beta

In [8]:
def errors(J):
    """Return (eps_d, eps_off, eps_t) for a tridiagonal matrix J.
    Handles NaN and Inf values gracefully without crashing."""
    import numpy as np
    # Convert SymPy matrix to a NumPy float array to bypass SymPy relational logic
    try:
        arr = np.array(J.tolist(), dtype=float)
    except:
        arr = np.array(J.evalf().tolist(), dtype=float)
        
    n = arr.shape
    
    # Extract diagonal and first off-diagonal
    d = np.abs(arr.diagonal())
    off = np.abs(1.0 - arr.diagonal(1))
    
    # Use nanmax/nansum: if a method fails (NaN), the table will show 'nan' instead of crashing
    eps_d   = np.nanmax(d) if d.size > 0 else 0.0
    eps_off = np.nanmax(off) if off.size > 0 else 0.0
    eps_t   = np.nansum(d) + np.nansum(off)
    
    return float(eps_d), float(eps_off), float(eps_t)

In [9]:
def make_inputs(n):
    """Build Lambda (reversed, largest first), c and beta for the
    tridiagonal matrix M with zeros on diagonal and ones off-diagonal."""
    Lambda = [2 * cos(k * pi / (n + 1)) for k in range(1, n + 1)]
    c      = [sin(k * pi / (n + 1)) / sqrt(sp.Rational(n + 1, 2))
              for k in range(1, n + 1)]
    # evaluate to Float so downstream arithmetic stays fast
    Lambda = [x.evalf() for x in Lambda]
    c      = [x.evalf() for x in c]
    Lambda.reverse()
    beta = mosertobeta(Lambda, c)   # does NOT mutate c
    return Lambda, c, beta

In [10]:
sizes = [10, 50, 100]#, 500, 1000]
rows  = []
 
for n in sizes:
    print(f"n = {n} ...", flush=True)
    Lambda, c, beta = make_inputs(n)
 
    T_rkpw  = rkpw(Lambda, c).evalf()
    T_invbi_c = invbi_cramer(Lambda, beta).evalf()
    T_invbi_g = invbi_gauss(Lambda, beta).evalf()
 
    for method, T in [("RKPW", T_rkpw), ("INVBI_Cramer", T_invbi_c), ("INVBI_Gauss", T_invbi_g)]:
        ed, eo, et = errors(T)
        rows.append({"$n$": n, "Method": method,
                     r"$\varepsilon_d$": ed,
                     r"$\varepsilon_{\mathrm{off}}$": eo,
                     r"$\varepsilon_t$": et})
 
df = pd.DataFrame(rows)
print(df.to_string(index=False))

n = 10 ...
n = 50 ...
n = 100 ...
 $n$       Method  $\varepsilon_d$  $\varepsilon_{\mathrm{off}}$  $\varepsilon_t$
  10         RKPW     2.220446e-15                  8.881784e-16     1.198694e-14
  10 INVBI_Cramer     5.107026e-15                  2.664535e-15     2.124689e-14
  10  INVBI_Gauss     1.776357e-15                  1.110223e-15     1.230266e-14
  50         RKPW     1.698641e-14                  1.021405e-14     2.265605e-13
  50 INVBI_Cramer     4.218292e-13                  2.395861e-13     2.942367e-12
  50  INVBI_Gauss     4.163336e-15                  3.774758e-15     9.255899e-14
 100         RKPW     4.840572e-14                  1.998401e-14     8.944978e-13
 100 INVBI_Cramer     7.396167e-13                  9.280354e-13     1.278862e-11
 100  INVBI_Gauss     1.071365e-14                  3.996803e-15     2.988827e-13


In [11]:
def fmt(x):
    """Format a float in LaTeX scientific notation, e.g. 1.23e-15 → 1.23 \times 10^{-15}."""
    if x == 0:
        return "$0$"
    s = f"{x:.2e}"
    mantissa, exp = s.split("e")
    exp = int(exp)
    return rf"${mantissa} \times 10^{{{exp}}}$"
 
df_latex = df.copy()
for col in [r"$\varepsilon_d$", r"$\varepsilon_{\mathrm{off}}$", r"$\varepsilon_t$"]:
    df_latex[col] = df_latex[col].apply(fmt)
 
latex_str = df_latex.to_latex(
    index=False,
    escape=False,
    column_format="rlccc",
    caption="Numerical errors for \\texttt{rkpw} and \\texttt{invbi} on the "
            "tridiagonal matrix $M$ with zeros on the diagonal and ones "
            "on the off-diagonal.",
    label="tab:errors",
)
print("\n" + latex_str)
 
# save to file for direct \input{} in LaTeX
with open("errors_table.tex", "w") as f:
    f.write(latex_str)
print("Table written to errors_table.tex")


\begin{table}
\centering
\caption{Numerical errors for \texttt{rkpw} and \texttt{invbi} on the tridiagonal matrix $M$ with zeros on the diagonal and ones on the off-diagonal.}
\label{tab:errors}
\begin{tabular}{rlccc}
\toprule
 $n$ &       Method &        $\varepsilon_d$ & $\varepsilon_{\mathrm{off}}$ &        $\varepsilon_t$ \\
\midrule
  10 &         RKPW & $2.22 \times 10^{-15}$ &       $8.88 \times 10^{-16}$ & $1.20 \times 10^{-14}$ \\
  10 & INVBI_Cramer & $5.11 \times 10^{-15}$ &       $2.66 \times 10^{-15}$ & $2.12 \times 10^{-14}$ \\
  10 &  INVBI_Gauss & $1.78 \times 10^{-15}$ &       $1.11 \times 10^{-15}$ & $1.23 \times 10^{-14}$ \\
  50 &         RKPW & $1.70 \times 10^{-14}$ &       $1.02 \times 10^{-14}$ & $2.27 \times 10^{-13}$ \\
  50 & INVBI_Cramer & $4.22 \times 10^{-13}$ &       $2.40 \times 10^{-13}$ & $2.94 \times 10^{-12}$ \\
  50 &  INVBI_Gauss & $4.16 \times 10^{-15}$ &       $3.77 \times 10^{-15}$ & $9.26 \times 10^{-14}$ \\
 100 &         RKPW & $4.84 \times 

/var/folders/j7/stm5sv8x15g7pb_5wh9zjywh0000gn/T/ipykernel_12872/829297431.py:14: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex_str = df_latex.to_latex(
